In [ ]:
import os, csv, math, random, copy, shutil, tempfile
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm import tqdm
from scipy.linalg import sqrtm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from torchvision.utils import save_image
from pytorch_msssim import ssim as ssim_loss

from sklearn.model_selection import train_test_split
from skimage.metrics import structural_similarity as ski_ssim
from skimage.metrics import peak_signal_noise_ratio as ski_psnr

os.environ.setdefault("TORCH_HOME", "/home/youruser/.cache/torch")

In [2]:
!pip install pytorch-msssim -q


In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
class Config:
    # Paths
    SAVE_DIR  = "/kaggle/working/Output"
    DATA_DIR  = "/kaggle/input/datasets/plhalvorsen/kvasir-v2-a-gastrointestinal-tract-dataset"
    

    # Architecture 
    IMG_SIZE   = 256
    IN_CH      = 3
    Z_DIM      = 256

    # Diffusion schedule
    T_STEPS         = 8
    BETA_MIN        = 0.1
    BETA_MAX        = 4.0
    IMG2IMG_NOISE_FRAC = 0.625

    # Training
    BATCH_SIZE      = 16
    NUM_WORKERS     = 2
    pin_memory      = True
    persistent_workers= False
    EPOCHS          = 500
    D_UPDATE_EVERY  = 1
    D_NOISE_INIT    = 0.05
    D_WARMUP_EPOCHS = 5

    # Optimiser 
    LR_G        = 2e-6
    LR_D        = 1.5e-6
    R1_GAMMA   = 0.02
    LAZY_REG   = 16
    EMA_DECAY  = 0.999

    # Loss weights 
    LAMBDA_ADV  = 2.0
    LAMBDA_REC  = 2.0
    LAMBDA_PERC = 2.5
    LAMBDA_JS   = 0.05

    # Discriminator balance
    D_LOSS_MIN  = 0.20
    D_LOSS_MAX  = 0.70
    REAL_LABEL  = 0.9
    FAKE_LABEL  = 0.1

    # Adaptive quality feedback
    ADAPTIVE_EVAL_EVERY  = 1
    ADAPTIVE_EVAL_N      = 96
    ADAPTIVE_WARMUP_EP   = 12
    ADAPTIVE_EMA         = 0.5
    ADAPTIVE_SSIM_INIT   = 0.50
    ADAPTIVE_PSNR_INIT   = 22.0
    ADAPTIVE_SSIM_TARGET = 0.85
    ADAPTIVE_PSNR_TARGET = 34.0

    # Logging / checkpointing
    FINAL_GEN_IMAGES  = 10000
    VIZ_EVERY         = 20
    VIS_SAMPLES       = 4
    PROGRESS_EVERY    = 10
    PROGRESS_SAMPLES  = 8
    SAVE_EVERY        = 20
    KEEP_LAST_N_CKPTS = 4

    RESUME           = True
    RESUME_PATH      = ""
    PHASE1_FREEZE_EP = 1
    PHASE2_UNFREEZE_EP = 60
    SEED             = 42
    DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
    SMOKE_TEST = False


cfg = Config()

if cfg.SMOKE_TEST:
    cfg.EPOCHS = 3
    cfg.BATCH_SIZE = 4
    cfg.FINAL_GEN_IMAGES = 16
    cfg.ADAPTIVE_EVAL_EVERY = 999
    cfg.VIZ_EVERY = 1
    cfg.PROGRESS_EVERY = 1
    cfg.SAVE_EVERY = 1
    cfg.D_WARMUP_EPOCHS = 0

print(f"Device      : {cfg.DEVICE}")
print(f"T={cfg.T_STEPS} | BETA_MAX={cfg.BETA_MAX}")
print(f"[v11] Fixed losses — adv={cfg.LAMBDA_ADV}  rec={cfg.LAMBDA_REC}  perc={cfg.LAMBDA_PERC}  js={cfg.LAMBDA_JS}")
print(f"[v11] Adaptive: SSIM/PSNR gap → G conditioning (every epoch)")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {p.total_memory/1e9:.1f} GB")


torch.backends.cudnn.benchmark = True     
torch.backends.cudnn.deterministic = False

## 3. Utility Functions

In [ ]:
def set_seed(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def tensor_to_numpy_uint8(t):
    return ((t.clamp(-1, 1) + 1.0) * 127.5).permute(0, 2, 3, 1).cpu().numpy().clip(0, 255).astype(np.uint8)

def tensor_01_to_numpy_uint8(t):
    return (t.clamp(0, 1) * 255.0).permute(0, 2, 3, 1).cpu().numpy().clip(0, 255).astype(np.uint8)

def tensor_to_01(t):
    return (t.clamp(-1, 1) + 1.0) * 0.5


def init_csv(path, header):
    if not Path(path).exists():
        with open(path, "w", newline="") as f:
            csv.writer(f).writerow(header)

def append_csv(path, row):
    with open(path, "a", newline="") as f:
        csv.writer(f).writerow(row)


def safe_torch_load(path, device):
    path = str(path)
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)
    except Exception as e:
        raise RuntimeError(f"Load failed: {path}\n{e}")

def setup_folders(root):
    root = Path(root)
    dirs = {
        "checkpoints": root / "checkpoints",
        "logs"        : root / "logs",
        "viz"         : root / "viz",
        "progress"    : root / "progress_gallery",
        "generated"   : root / "generated_images",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs

def disk_usage_gb(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1e9


def js_divergence(fake, real, bins=32, eps=1e-8):

    fake = ((fake.clamp(-1, 1) + 1.0) / 2.0)
    real = ((real.detach().clamp(-1, 1) + 1.0) / 2.0)

    fake = F.interpolate( fake, size=(64, 64),mode="bilinear", align_corners=False)

    real = F.interpolate(real, size=(64, 64), mode="bilinear", align_corners=False)

    fake_values = fake.reshape(-1)
    real_values = real.reshape(-1)

    centers = torch.linspace(
        0.0, 1.0, bins,
        device=fake.device,
        dtype=fake.dtype
    )

    bin_width = 1.0 / (bins - 1)
    fake_weights = torch.relu(
        1.0 - torch.abs(
            fake_values.unsqueeze(1) - centers.unsqueeze(0)
        ) / bin_width
    )

    real_weights = torch.relu(
        1.0 - torch.abs(
            real_values.unsqueeze(1) - centers.unsqueeze(0)
        ) / bin_width
    )

    fake_weights = fake_weights / (
        fake_weights.sum(dim=1, keepdim=True) + eps
    )

    real_weights = real_weights / (
        real_weights.sum(dim=1, keepdim=True) + eps
    )

    p = fake_weights.mean(dim=0)
    q = real_weights.mean(dim=0)

    p = p / (p.sum() + eps)
    q = q / (q.sum() + eps)

    m = 0.5 * (p + q)

    kl_pm = torch.sum(
        p * torch.log((p + eps) / (m + eps))
    )

    kl_qm = torch.sum(
        q * torch.log((q + eps) / (m + eps))
    )

    js = 0.5 * (kl_pm + kl_qm)

    return js

## 4. Dataset & Data Loaders

In [ ]:
class KvasirDataset(Dataset):

    def __init__(self, data_dir, img_size=256):
        self.img_size    = img_size
        self.image_paths = []
        self.labels      = []

        class_names = sorted([d for d in os.listdir(data_dir)
                               if os.path.isdir(os.path.join(data_dir, d))])
        self.class_to_idx = {c: i for i, c in enumerate(class_names)}
        self.idx_to_class = {i: c for c, i in self.class_to_idx.items()}

        for cls in class_names:
            cls_dir = os.path.join(data_dir, cls)
            subs    = [d for d in os.listdir(cls_dir)
                       if os.path.isdir(os.path.join(cls_dir, d))]
            if len(subs) == 1:
                cls_dir = os.path.join(cls_dir, subs[0])
            for f in os.listdir(cls_dir):
                if f.lower().endswith((".png", ".jpg", ".jpeg")):
                    self.image_paths.append(os.path.join(cls_dir, f))
                    self.labels.append(self.class_to_idx[cls])

        self.base_tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3),
        ])
        self._mode = "train"
        print(f"Dataset: {len(self.image_paths)} images | {len(class_names)} classes")

    def set_mode(self, m): self._mode = m
    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.image_paths[idx]).convert("RGB")
        except:
            img = Image.new("RGB", (self.img_size, self.img_size))
        return self.base_tf(img), self.labels[idx]


def build_splits(data_dir, img_size, train_ratio=0.70, val_ratio=0.15, seed=42):
    full_ds = KvasirDataset(data_dir, img_size)
    idx     = np.arange(len(full_ds))
    lbl     = np.array(full_ds.labels)

    tr, tmp = train_test_split(idx, test_size=1 - train_ratio,
                               stratify=lbl, random_state=seed)
    vl, te  = train_test_split(tmp,
                               test_size=1 - val_ratio / (1 - train_ratio),
                               stratify=lbl[tmp], random_state=seed)
    print(f"Split → Train:{len(tr)}  Val:{len(vl)}  Test:{len(te)}")
    return full_ds, Subset(full_ds, tr), Subset(full_ds, vl), Subset(full_ds, te)


## 5. VP-SDE Diffusion Schedule

In [ ]:
def build_ddgan_schedule(T, beta_min=0.1, beta_max=5.0, device="cpu"):
    t_idx  = torch.arange(1, T + 1, dtype=torch.float64, device=device)
    t_norm = t_idx / T

    def sigma2(tp):
        return 1.0 - torch.exp(-beta_min * tp - 0.5 * (beta_max - beta_min) * tp ** 2)

    s2_t   = sigma2(t_norm)
    s2_tm1 = sigma2((t_idx - 1) / T)
    ab_t   = (1.0 - s2_t).clamp(min=1e-8)
    ab_tm1 = (1.0 - s2_tm1).clamp(min=1e-8)
    beta_t = (1.0 - ab_t / ab_tm1).clamp(1e-6, 0.999)
    alpha_t = 1.0 - beta_t
    post_var = (s2_tm1 / s2_t.clamp(min=1e-8) * beta_t).clamp(min=1e-8)
    ab_full  = torch.cat([torch.ones(1, dtype=torch.float64, device=device), ab_t])

    sched = {
        "T"        : T,
        "beta_t"   : beta_t.float(),
        "alpha_t"  : alpha_t.float(),
        "alpha_bar": ab_full.float(),
        "ab_tm1"   : ab_tm1.float(),
        "post_var" : post_var.float(),
    }

    print(f"\nVP-SDE schedule (T={T}, BETA_MAX={beta_max}):")
    for t in range(1, T + 1):
        noise = 100 * (1 - ab_full[t].item())
        step  = (ab_full[t - 1] - ab_full[t]).item() * 100
        print(f"  t={t:2d}: noise={noise:5.1f}%  step={step:5.1f}%")
    return sched


def q_sample(x0, t, sched):
    ab  = sched["alpha_bar"][t].view(-1, 1, 1, 1)
    eps = torch.randn_like(x0)
    return torch.sqrt(ab) * x0 + torch.sqrt(1.0 - ab) * eps, eps

def q_sample_pair(x0, t, sched):
    ab_t   = sched["alpha_bar"][t].view(-1, 1, 1, 1)
    ab_tm1 = sched["alpha_bar"][(t - 1).clamp(min=0)].view(-1, 1, 1, 1)
    eps_t, eps_tm1 = torch.randn_like(x0), torch.randn_like(x0)
    return (
        torch.sqrt(ab_tm1) * x0 + torch.sqrt(1.0 - ab_tm1) * eps_tm1,
        torch.sqrt(ab_t)   * x0 + torch.sqrt(1.0 - ab_t)   * eps_t,
        eps_tm1, eps_t,
    )

def posterior_sample(x_t, x0_hat, t, sched, add_noise=True):
    beta_t   = sched["beta_t"][t - 1].view(-1, 1, 1, 1)
    alpha_t  = sched["alpha_t"][t - 1].view(-1, 1, 1, 1)
    ab_t     = sched["alpha_bar"][t].view(-1, 1, 1, 1)
    ab_tm1   = sched["alpha_bar"][(t - 1).clamp(min=0)].view(-1, 1, 1, 1)
    post_var = sched["post_var"][t - 1].view(-1, 1, 1, 1)
    mu = (
        (torch.sqrt(ab_tm1) * beta_t / (1.0 - ab_t).clamp(1e-8)) * x0_hat
        + (torch.sqrt(alpha_t) * (1.0 - ab_tm1) / (1.0 - ab_t).clamp(1e-8)) * x_t
    )
    return mu + torch.sqrt(post_var) * torch.randn_like(x_t) if add_noise else mu


def sinusoidal_embedding(t, dim):
    half  = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device, dtype=torch.float32) / half)
    args  = t.float().unsqueeze(1) * freqs.unsqueeze(0)
    return torch.cat([torch.sin(args), torch.cos(args)], dim=1)

##  6. Loss Functions

In [ ]:
def reconstruction_loss(fake, real):
    f = (fake.clamp(-1, 1) + 1.0) * 0.5
    r = (real.clamp(-1, 1) + 1.0) * 0.5
    return F.l1_loss(f, r)

class PerceptualLoss(nn.Module):

    def __init__(self):
        super().__init__()
        try:
            vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features.eval()
            print("  PerceptualLoss: VGG19 loaded (pretrained)")
        except:
            vgg = models.vgg19(weights=None).features.eval()
            print("  VGG19: random weights")

        self.s1 = nn.Sequential(*list(vgg[:4]))
        self.s2 = nn.Sequential(*list(vgg[4:9]))
        self.s3 = nn.Sequential(*list(vgg[9:18]))
        self.s4 = nn.Sequential(*list(vgg[18:27]))
        for p in self.parameters():
            p.requires_grad = False

        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.w = [1.0, 0.75, 0.5, 0.25]

    def _prep(self, x):
        return ((x.clamp(-1, 1) + 1.0) * 0.5 - self.mean) / self.std

    def forward(self, fake, real):
        f = self._prep(fake.float())
        r = self._prep(real.float().detach())
        loss = 0.0
        for layer, w in zip([self.s1, self.s2, self.s3, self.s4], self.w):
            f = layer(f); r = layer(r)
            loss += w * F.mse_loss(f, r.detach())
        return loss / 10.0 

## 7. Generator Architecture



In [ ]:
def _replace_bn_gn(module, num_groups=8):
    for name, child in module.named_children():
        if isinstance(child, nn.BatchNorm2d):
            setattr(module, name,
                    nn.GroupNorm(min(num_groups, child.num_features), child.num_features))
        else:
            _replace_bn_gn(child, num_groups)

def _sn(m): return nn.utils.spectral_norm(m)


class QualityFeedbackEmbedding(nn.Module):
    def __init__(self, cond_dim):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, cond_dim))

    def forward(self, ssim_gap, psnr_gap, batch_size, device):
        vec = torch.tensor(
            [[float(ssim_gap) * 2.0, float(psnr_gap) / 10.0]],
            dtype=torch.float32, device=device,
        ).expand(batch_size, -1)
        return self.mlp(vec)


class CBAMAttention(nn.Module):
    def __init__(self, ch, reduction=8):
        super().__init__()
        mid = max(ch // reduction, 16)
        self.ch_mlp  = nn.Sequential(nn.Linear(ch, mid, bias=False), nn.ReLU(),
                                     nn.Linear(mid, ch, bias=False))
        self.sp_conv = nn.Sequential(nn.Conv2d(2, 1, 7, padding=3, bias=False), nn.Sigmoid())

    def forward(self, x):
        b, c, h, w = x.shape
        ch_attn = torch.sigmoid(
            self.ch_mlp(x.mean(dim=[2, 3])) + self.ch_mlp(x.amax(dim=[2, 3]))
        )
        x = x * ch_attn.view(b, c, 1, 1)
        sp_attn = self.sp_conv(torch.cat(
            [x.mean(dim=1, keepdim=True), x.amax(dim=1, keepdim=True)], dim=1
        ))
        return x * sp_attn


class BottleneckResBlock(nn.Module):
    def __init__(self, ch, cond_dim):
        super().__init__()
        neck = max(ch // 4, 16)
        self.norm1  = nn.GroupNorm(min(8, ch), ch)
        self.conv1  = nn.Conv2d(ch, neck, 1, bias=False)
        self.norm2  = nn.GroupNorm(min(8, neck), neck)
        self.conv2  = nn.Conv2d(neck, neck, 3, padding=1, bias=False)
        self.norm3  = nn.GroupNorm(min(8, neck), neck)
        self.conv3  = nn.Conv2d(neck, ch, 1, bias=False)
        self.cond_proj = nn.Sequential(nn.ReLU(), nn.Linear(cond_dim, neck * 2))

    def forward(self, x, cond):
        h  = F.relu(self.norm1(x))
        h  = self.conv1(h)
        sc, sh = self.cond_proj(cond).chunk(2, dim=1)
        h  = self.norm2(h) * (1 + sc[:, :, None, None]) + sh[:, :, None, None]
        h  = F.relu(h)
        h  = self.conv2(h)
        h  = F.relu(self.norm3(h))
        h  = self.conv3(h)
        return x + h


# Decoder Up-Blocks 
class ResUpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim):
        super().__init__()
        self.up   = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
        )
        self.res1 = BottleneckResBlock(out_ch, cond_dim)
        self.res2 = BottleneckResBlock(out_ch, cond_dim)
        self.res3 = BottleneckResBlock(out_ch, cond_dim)

    def forward(self, x, cond):
        return self.res3(self.res2(self.res1(self.up(x), cond), cond), cond)


class ResUpBlockWithAttn(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim, use_attn=False):
        super().__init__()
        self.up   = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
        )
        self.res1 = BottleneckResBlock(out_ch, cond_dim)
        self.res2 = BottleneckResBlock(out_ch, cond_dim)
        self.res3 = BottleneckResBlock(out_ch, cond_dim)
        self.attn = CBAMAttention(out_ch) if use_attn else nn.Identity()

    def forward(self, x, cond):
        x = self.res3(self.res2(self.res1(self.up(x), cond), cond), cond)
        return self.attn(x)


# FPN Lateral Fusion
class FPNFusion(nn.Module):
    def __init__(self, enc_ch, top_ch, out_ch, fpn_dim=256):
        super().__init__()
        self.lateral  = nn.Conv2d(enc_ch, fpn_dim, 1, bias=False)
        self.top_proj = nn.Conv2d(top_ch, fpn_dim, 1, bias=False)
        self.fuse     = nn.Sequential(
            nn.Conv2d(fpn_dim * 2, fpn_dim, 1, bias=False),
            nn.GroupNorm(min(8, fpn_dim), fpn_dim), nn.ReLU(),
            nn.Conv2d(fpn_dim, out_ch, 3, padding=1, bias=False),
        )

    def forward(self, enc_feat, top_feat):
        lat = self.lateral(enc_feat)
        if top_feat.shape[2:] != enc_feat.shape[2:]:
            top_feat = F.interpolate(top_feat, enc_feat.shape[2:],
                                     mode="bilinear", align_corners=False)
        return self.fuse(torch.cat([lat, self.top_proj(top_feat)], dim=1))

In [ ]:
class DenseNet121Generator(nn.Module):


    def __init__(self, in_ch=3, out_ch=3, t_dim=512, z_dim=256):
        super().__init__()
        self.t_dim = t_dim
        self.z_dim = z_dim
        cond_dim   = t_dim * 4

        # Conditioning embeddings
        self.t_embed = nn.Sequential(nn.Linear(t_dim, cond_dim), nn.ReLU(),
                                     nn.Linear(cond_dim, cond_dim), nn.ReLU())
        self.z_embed = nn.Sequential(nn.Linear(z_dim, cond_dim), nn.ReLU(),
                                     nn.Linear(cond_dim, cond_dim), nn.ReLU(),
                                     nn.Linear(cond_dim, cond_dim), nn.ReLU())
        self.q_embed = QualityFeedbackEmbedding(cond_dim)

        # Scale/shift projections per encoder stage
        self.t_stem = nn.Linear(cond_dim, 64 * 2)
        self.t_l1   = nn.Linear(cond_dim, 256 * 2)
        self.t_l2   = nn.Linear(cond_dim, 512 * 2)
        self.t_l3   = nn.Linear(cond_dim, 1024 * 2)

        # DenseNet-121 encoder 
        base = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        self.enc_stem   = nn.Sequential(base.features.conv0, base.features.norm0,
                                        base.features.relu0, base.features.pool0)
        self.enc_block1 = base.features.denseblock1
        self.enc_trans1 = base.features.transition1
        self.enc_block2 = base.features.denseblock2
        self.enc_trans2 = base.features.transition2
        self.enc_block3 = base.features.denseblock3
        self.enc_norm   = nn.GroupNorm(8, 1024)
        for m in [self.enc_stem, self.enc_block1, self.enc_trans1,
                  self.enc_block2, self.enc_trans2, self.enc_block3]:
            _replace_bn_gn(m)

        # Decoder
        self.dec1 = ResUpBlock(1024, 512, cond_dim)
        self.dec2 = ResUpBlockWithAttn(512, 256, cond_dim, use_attn=True)
        self.dec3 = ResUpBlockWithAttn(256, 128, cond_dim, use_attn=True)
        self.dec4 = ResUpBlock(128, 64, cond_dim)

        # FPN lateral connections
        self.fpn2 = FPNFusion(enc_ch=512,  top_ch=1024, out_ch=512, fpn_dim=256)
        self.fpn1 = FPNFusion(enc_ch=256,  top_ch=512,  out_ch=256, fpn_dim=128)
        self.fpn0 = FPNFusion(enc_ch=64,   top_ch=256,  out_ch=128, fpn_dim=64)

        # Shallow feature path
        self.enc_shallow = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1, bias=False), nn.GroupNorm(8, 32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=1, padding=1, bias=False), 
            nn.GroupNorm(8, 64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1, bias=False), nn.GroupNorm(8, 64), nn.ReLU(),
        )
        self.fpn_shallow = FPNFusion(enc_ch=64, top_ch=128, out_ch=64, fpn_dim=64)

        self.out_refine = nn.Sequential(
            nn.GroupNorm(8, 64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1, bias=False), nn.GroupNorm(8, 64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1, bias=False), nn.GroupNorm(8, 64), nn.ReLU(),
            nn.Conv2d(64, 32, 3, padding=1, bias=False), nn.GroupNorm(8, 32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1, bias=False), nn.GroupNorm(8, 32), nn.ReLU(),
        )
        self.out_conv = nn.Conv2d(32, out_ch, 1)

    def _mod(self, x, proj, cond, ch):
        sc, sh = proj(cond).chunk(2, dim=1)
        return x * (1 + sc.view(-1, ch, 1, 1)) + sh.view(-1, ch, 1, 1)

    def forward(self, x_t, z, t, ssim_gap=0.45, psnr_gap=18.0):
        H, W = x_t.shape[2], x_t.shape[3]; b = x_t.size(0)
        cond = self.t_embed(sinusoidal_embedding(t, self.t_dim)) + self.z_embed(z) + self.q_embed(ssim_gap, psnr_gap, b, x_t.device)
        e0 = self._mod(self.enc_stem(x_t), self.t_stem, cond, 64)
        e1 = self._mod(self.enc_block1(e0), self.t_l1, cond, 256); x = self.enc_trans1(e1)
        e2 = self._mod(self.enc_block2(x), self.t_l2, cond, 512); x = self.enc_trans2(e2)
        bn = self._mod(F.relu(self.enc_norm(self.enc_block3(x))), self.t_l3, cond, 1024)
        shallow = self.enc_shallow(x_t)
        d1 = self.dec1(bn, cond); d1 = d1 + self.fpn2(e2, bn)
        d2 = self.dec2(d1, cond); d2 = d2 + self.fpn1(e1, d1)
        d3 = self.dec3(d2, cond)
        fpn0_out = self.fpn0(e0, d2)
        if fpn0_out.shape[2:] != d3.shape[2:]:
            fpn0_out = F.interpolate(fpn0_out, d3.shape[2:], mode="bilinear", align_corners=False)
        d3 = d3 + fpn0_out
        d4 = self.dec4(d3, cond)
        fpn_s_out = self.fpn_shallow(shallow, d3)
        if fpn_s_out.shape[2:] != d4.shape[2:]:
            fpn_s_out = F.interpolate(fpn_s_out, d4.shape[2:], mode="bilinear", align_corners=False)
        d4 = d4 + fpn_s_out

        out = 0.95 * torch.tanh(self.out_conv(self.out_refine(d4)))
        if out.shape[2] != H or out.shape[3] != W:
            out = F.interpolate(out, (H, W), mode="bilinear", align_corners=False)
        return out

## 8. Discriminator Architecture

In [ ]:
class StrongerDiscriminator(nn.Module):


    def __init__(self, in_ch=3, t_dim=512):
        super().__init__()
        self.t_dim = t_dim
        ndf = 128

        self.t_embed = nn.Sequential(nn.Linear(t_dim, 512), nn.ReLU(),
                                     nn.Linear(512, 512), nn.ReLU())
        self.t_s1 = nn.Linear(512, ndf)
        self.t_s2 = nn.Linear(512, ndf * 2)
        self.t_s3 = nn.Linear(512, ndf * 4)

        self.stem  = nn.Sequential(_sn(nn.Conv2d(in_ch * 2, ndf, 4, 2, 1)),
                                   nn.LeakyReLU(0.2, inplace=True))
        self.s1    = nn.Sequential(
            _sn(nn.Conv2d(ndf, ndf, 3, 1, 1)), nn.GroupNorm(min(8, ndf), ndf),
            nn.LeakyReLU(0.2, inplace=True),
            _sn(nn.Conv2d(ndf, ndf, 3, 1, 1)), nn.GroupNorm(min(8, ndf), ndf),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.down2 = _sn(nn.Conv2d(ndf, ndf * 2, 4, 2, 1))
        self.s2    = nn.Sequential(
            nn.GroupNorm(min(8, ndf * 2), ndf * 2), nn.LeakyReLU(0.2, inplace=True),
            _sn(nn.Conv2d(ndf * 2, ndf * 2, 3, 1, 1)), nn.GroupNorm(min(8, ndf * 2), ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            _sn(nn.Conv2d(ndf * 2, ndf * 2, 3, 1, 1)), nn.GroupNorm(min(8, ndf * 2), ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.patch_head = _sn(nn.Conv2d(ndf * 2, 1, 4, 1, 1))
        self.down3      = _sn(nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1))
        self.s3         = nn.Sequential(
            nn.GroupNorm(min(8, ndf * 4), ndf * 4), nn.LeakyReLU(0.2, inplace=True),
            _sn(nn.Conv2d(ndf * 4, ndf * 4, 3, 1, 1)), nn.GroupNorm(min(8, ndf * 4), ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            _sn(nn.Conv2d(ndf * 4, ndf * 4, 3, 1, 1)), nn.GroupNorm(min(8, ndf * 4), ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.pool        = nn.AdaptiveAvgPool2d(1)
        self.global_head = nn.Sequential(
            nn.Flatten(),
            _sn(nn.Linear(ndf * 4 + 512, 512)), nn.LeakyReLU(0.2),
            _sn(nn.Linear(512, 1)),
        )
        self.combine = _sn(nn.Linear(2, 1, bias=False))

    def forward(self, x_tm1, x_t, t):
        t_emb = self.t_embed(sinusoidal_embedding(t, self.t_dim))
        inp   = torch.cat([x_tm1, x_t], dim=1)
        f     = self.stem(inp)
        f     = self.s1(f + self.t_s1(t_emb).view(-1, f.shape[1], 1, 1))
        f     = self.down2(f)
        f     = self.s2(f + self.t_s2(t_emb).view(-1, f.shape[1], 1, 1))
        p_patch  = self.patch_head(f).mean(dim=[1, 2, 3], keepdim=False)
        f     = self.down3(f)
        f     = self.s3(f + self.t_s3(t_emb).view(-1, f.shape[1], 1, 1))
        gf    = self.pool(f).flatten(1)
        p_global = self.global_head(torch.cat([gf, t_emb], dim=1)).squeeze(1)
        return self.combine(torch.stack([p_patch, p_global], dim=1))


## 9. EMA, Quality State & Checkpointing

In [ ]:
class QualityState:

    def __init__(self, cfg):
        self.ssim        = cfg.ADAPTIVE_SSIM_INIT
        self.psnr        = cfg.ADAPTIVE_PSNR_INIT
        self.ema         = cfg.ADAPTIVE_EMA
        self.target_ssim = cfg.ADAPTIVE_SSIM_TARGET
        self.target_psnr = cfg.ADAPTIVE_PSNR_TARGET

    def update(self, ms, mp):
        self.ssim = self.ema * self.ssim + (1.0 - self.ema) * ms
        self.psnr = self.ema * self.psnr + (1.0 - self.ema) * mp

    def get_gap(self):
        return self.target_ssim - self.ssim, self.target_psnr - self.psnr

    def get_measured(self):
        return self.ssim, self.psnr

    def state_dict(self):
        return {"ssim": self.ssim, "psnr": self.psnr}

    def load_state_dict(self, d):
        self.ssim = d.get("ssim", self.ssim)
        self.psnr = d.get("psnr", self.psnr)


class EMA:

    def __init__(self, model, decay=0.9999):
        self.decay  = decay
        self.shadow = copy.deepcopy(model)
        self.shadow.eval()
        for p in self.shadow.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update(self, model):
        for sp, mp in zip(self.shadow.parameters(), model.parameters()):
            sp.data.mul_(self.decay).add_(mp.data, alpha=1.0 - self.decay)

    def state_dict(self):
        return self.shadow.state_dict()

    def load_state_dict(self, sd):
        self.shadow.load_state_dict(sd)



def safe_torch_load(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=device)


def save_ckpt(path, ep, G, D, opt_g, opt_d, sched_g, sched_d, qs, best_fid, best_ep, ema=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    data = {
        "tag":           CURRENT_TAG,
        "ep":            ep,
        "G":             G.state_dict(),
        "D":             D.state_dict(),
        "opt_g":         opt_g.state_dict(),
        "opt_d":         opt_d.state_dict(),
        "sched_g":       sched_g.state_dict(),
        "sched_d":       sched_d.state_dict(),
        "quality_state": qs.state_dict(),
        "best_fid":      best_fid,
        "best_ep":       best_ep,
    }

    if ema is not None:
        data["ema_G"] = ema.state_dict()

    fd, tmp = tempfile.mkstemp(dir=path.parent, suffix=".tmp")
    os.close(fd)
    try:
        torch.save(data, tmp)
        shutil.move(tmp, str(path))
    except Exception:
        os.unlink(tmp)
        raise

    print(f"  Saved → {path.name}  ({path.stat().st_size / 1e6:.0f} MB)")


def load_ckpt(path, G, D, opt_g, opt_d, sched_g, sched_d, qs, device, ema=None,
              fallback_lr_g=None, fallback_lr_d=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {path}")

    print(f"Loading: {path}")
    ck = safe_torch_load(path, device)

    def _partial(model, sd, name):
        ms      = model.state_dict()
        ld      = {k: v for k, v in sd.items() if k in ms and v.shape == ms[k].shape}
        skipped = len(ms) - len(ld)
        model.load_state_dict(ld, strict=False)
        print(f"  {name}: {len(ld)}/{len(ms)} keys loaded"
              + (f"  ({skipped} new/mismatched — random init)" if skipped else ""))

    _partial(G, ck["G"], "G")
    _partial(D, ck["D"], "D")

    try:
        opt_g.load_state_dict(ck["opt_g"])
        opt_d.load_state_dict(ck["opt_d"])
        print(f"  Optimizers: restored  "
              f"(G lr={opt_g.param_groups[0]['lr']:.2e}  "
              f"D lr={opt_d.param_groups[0]['lr']:.2e})")
    except (KeyError, ValueError, RuntimeError) as e:
        print(f"  WARNING — optimizer restore failed: {e}. Optimizers will restart from scratch.")

    if "sched_g" in ck:
        sched_g.load_state_dict(ck["sched_g"])
        try:
            print(f"  sched_g: restored  last_lr={sched_g.get_last_lr()}")
        except Exception:
            print("  sched_g: restored")
    else:
        print("  WARNING — sched_g not in checkpoint (old v11 format).")
        if fallback_lr_g is not None:
            for pg in opt_g.param_groups:
                pg["lr"] = fallback_lr_g
            print(f"            G LR manually set to {fallback_lr_g:.2e}")
        else:
            print("            Pass fallback_lr_g=<value> to avoid wrong LR on resume.")

    if "sched_d" in ck:
        sched_d.load_state_dict(ck["sched_d"])
        try:
            print(f"  sched_d: restored  last_lr={sched_d.get_last_lr()}")
        except Exception:
            print("  sched_d: restored")
    else:
        print("  WARNING — sched_d not in checkpoint (old v11 format).")
        if fallback_lr_d is not None:
            for pg in opt_d.param_groups:
                pg["lr"] = fallback_lr_d
            print(f"            D LR manually set to {fallback_lr_d:.2e}")
        else:
            print("  WARNING — wrong LR on resume.")
    
    if ema is not None:
        if "ema_G" in ck:
            _partial(ema.shadow, ck["ema_G"], "EMA_G")
        else:
            print("  WARNING — ema_G not in checkpoint; EMA starts from current G.")

    if "quality_state" in ck:
        qs.load_state_dict(ck["quality_state"])
        print(f"  QualityState: SSIM={qs.ssim:.4f}  PSNR={qs.psnr:.2f}")
    else:
        print("  QualityState: not found, using defaults.")

    start_ep = ck["ep"] + 1
    best_fid = ck.get("best_fid", float("inf"))
    best_ep  = ck.get("best_ep", 0)

    return start_ep, best_fid, best_ep

## 10. Inference & Visualisation Helpers

In [ ]:
@torch.no_grad()
def ddgan_img2img(G, sched, cfg, src_imgs, device, noise_frac=None, ssim_gap=0.0, psnr_gap=0.0):
    G.eval()
    nf = noise_frac if noise_frac is not None else cfg.IMG2IMG_NOISE_FRAC  # FIX-2: "or" fails for 0.0
    T  = sched["T"]
    src = src_imgs.to(device)
    b   = src.size(0)
    t_start = max(1, int(nf * T))
    x, _    = q_sample(src, torch.full((b,), t_start, device=device, dtype=torch.long), sched)
    for tv in range(t_start, 0, -1):
        tb = torch.full((b,), tv, device=device, dtype=torch.long)
        x  = posterior_sample(
            x,
            G(x, torch.randn(b, cfg.Z_DIM, device=device), tb, ssim_gap=ssim_gap, psnr_gap=psnr_gap),
            tb, sched, add_noise=(tv > 1),
        )
    return tensor_to_01(x.clamp(-1, 1))


@torch.no_grad()
def save_progress(G, sched, cfg, src_imgs, ep, out_dir, device, sg=0.0, pg=0.0):
    G.eval()
    n   = min(cfg.PROGRESS_SAMPLES, src_imgs.size(0))
    src = src_imgs[:n].to(device)
    panel = torch.cat([
        tensor_to_01(src).cpu(),
        ddgan_img2img(G, sched, cfg, src, device, ssim_gap=sg, psnr_gap=pg).cpu(),
    ], dim=0)
    save_image(panel, Path(out_dir) / f"ep{ep:03d}.png", nrow=n, padding=2, pad_value=0.5)


@torch.no_grad()
def save_viz(G, sched, cfg, imgs, ep, out_dir, device, sg=0.0, pg=0.0):
    G.eval()
    n  = min(cfg.VIS_SAMPLES, imgs.size(0))
    x0 = imgs[:n].to(device)
    T  = sched["T"]
    rows = [tensor_to_01(x0).cpu()]
    for tv in range(1, T + 1, max(1, T // 4)):
        xt, _ = q_sample(x0, torch.full((n,), tv, device=device, dtype=torch.long), sched)
        rows.append(tensor_to_01(xt.clamp(-1, 1)).cpu())
    tb = torch.full((n,), T, device=device, dtype=torch.long)
    xt, _ = q_sample(x0, tb, sched)
    rows.append(tensor_to_01(
        G(xt, torch.randn(n, cfg.Z_DIM, device=device), tb, ssim_gap=sg, psnr_gap=pg).clamp(-1, 1)
    ).cpu())
    rows.append(ddgan_img2img(G, sched, cfg, x0, device, ssim_gap=sg, psnr_gap=pg).cpu())
    save_image(torch.cat(rows, 0), Path(out_dir) / f"ep{ep:03d}.png",
               nrow=n, padding=2, pad_value=0.3)


## 11. Evaluation Metrics (SSIM, PSNR, FID)

In [ ]:
def compute_ssim_psnr(gen, src):
    gn = tensor_01_to_numpy_uint8(gen)
    sn = tensor_01_to_numpy_uint8(src)
    ss, ps = [], []
    for g, s in zip(gn, sn):
        ss.append(ski_ssim(s, g, channel_axis=2, data_range=255))
        ps.append(ski_psnr(s, g, data_range=255))
    return float(np.mean(ss)), float(np.mean(ps))


class FIDEvaluator:

    def __init__(self, device):
        self.device = device
        self.rf, self.ff = [], []
        try:
            inc = models.inception_v3(weights=models.Inception_V3_Weights.DEFAULT)
            print("  FID: InceptionV3 loaded (pretrained)")
        except:
            inc = models.inception_v3(weights=None)
        inc.aux_logits = False
        inc.AuxLogits  = None
        self.pool = nn.Sequential(
            inc.Conv2d_1a_3x3, inc.Conv2d_2a_3x3, inc.Conv2d_2b_3x3, nn.MaxPool2d(3, 2),
            inc.Conv2d_3b_1x1, inc.Conv2d_4a_3x3, nn.MaxPool2d(3, 2),
            inc.Mixed_5b, inc.Mixed_5c, inc.Mixed_5d,
            inc.Mixed_6a, inc.Mixed_6b, inc.Mixed_6c, inc.Mixed_6d, inc.Mixed_6e,
            inc.Mixed_7a, inc.Mixed_7b, inc.Mixed_7c,
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        ).to(device).eval()
        for p in self.pool.parameters():
            p.requires_grad = False
        self.norm   = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        self.resize = transforms.Resize((299, 299), antialias=True)

    @torch.no_grad()
    def _feat(self, imgs_01):
        return self.pool(self.norm(self.resize(imgs_01.to(self.device)))).cpu().numpy()

    def reset(self):             self.rf.clear();  self.ff.clear()
    def add_real(self, i):       self.rf.append(self._feat(i))
    def add_fake(self, i):       self.ff.append(self._feat(i))

    def compute(self):
        r = np.concatenate(self.rf, 0)
        f = np.concatenate(self.ff, 0)
        mu1, s1 = r.mean(0), np.cov(r, rowvar=False)
        mu2, s2 = f.mean(0), np.cov(f, rowvar=False)
        diff = mu1 - mu2
        eps = 1e-6  
        offset = np.eye(s1.shape[0]) * eps
        cm   = sqrtm((s1 + offset) @ (s2 + offset))
        if np.iscomplexobj(cm):
            cm = cm.real
        return float(diff @ diff + np.trace(s1 + s2 - 2 * cm))

##  12. Final Generation (Equal Images per Class)

In [ ]:
@torch.no_grad()
def generate_equal_per_class(G, sched, cfg, full_ds, train_indices,out_dir, device, sg, pg):
    
    G.eval()
    idx_to_class = {
        v: k for k, v in full_ds.class_to_idx.items()
    }

    n_classes = len(idx_to_class)
    base_target = cfg.FINAL_GEN_IMAGES // n_classes
    remainder = cfg.FINAL_GEN_IMAGES % n_classes

    out_dir = Path(out_dir)
    for class_name in idx_to_class.values():
        (out_dir / class_name).mkdir(
            parents=True, exist_ok=True
        )

    class_to_indices = {
        name: [] for name in idx_to_class.values()
    }

    for idx in train_indices:
        label = full_ds.labels[idx]
        class_name = idx_to_class[label]
        class_to_indices[class_name].append(idx)

    for class_name, indices in class_to_indices.items():
        if len(indices) == 0:
            raise RuntimeError(
                f"No training images available for {class_name}"
            )

    print("\n" + "=" * 60)
    print(f"Target images: {cfg.FINAL_GEN_IMAGES}")
    print("=" * 60)

    cc = {}
    total_generated = 0

    for rank, class_name in enumerate(sorted(class_to_indices)):

        target = base_target + (
            1 if rank < remainder else 0
        )

        source_indices = class_to_indices[class_name]

        selected_indices = [
            random.choice(source_indices)
            for _ in range(target)
        ]

        print(
            f"[{rank+1}/{n_classes}] {class_name}: "
            f"target={target}, training pool={len(source_indices)}"
        )

        loader = DataLoader(
            Subset(full_ds, selected_indices),
            batch_size=cfg.BATCH_SIZE,
            shuffle=False,
            num_workers=cfg.NUM_WORKERS,
            pin_memory=True,
            drop_last=False
        )

        count = 0

        for imgs, _ in loader:

            imgs = imgs.to(device, non_blocking=True)

            generated = ddgan_img2img(
                G, sched, cfg, imgs, device,
                ssim_gap=sg,
                psnr_gap=pg
            )

            for image in generated:
                if count >= target:
                    break

                save_image(
                    image,
                    out_dir / class_name / f"gen_{count:05d}.png"
                )
                count += 1

        if count != target:
            raise RuntimeError(
                f"{class_name}: generated {count}, expected {target}"
            )

        cc[class_name] = count
        total_generated += count

        print(f"Saved: {count}/{target}")

    print("\n" + "=" * 60)
    print(f"Total generated: {total_generated}")
    print("=" * 60)

    return cc

In [ ]:
import torch
ck = torch.load("/kaggle/working/checkpoints/ep320.pt", map_location="cpu")
ck_keys = set(ck["G"].keys())

G_tmp = DenseNet121Generator(cfg.IN_CH, cfg.IN_CH, t_dim=512, z_dim=cfg.Z_DIM)
model_keys = set(G_tmp.state_dict().keys())

print("Keys in model but NOT in checkpoint (will be random init):")
for k in sorted(model_keys - ck_keys):
    print(f"  {k}  shape={G_tmp.state_dict()[k].shape}")

print("\nKeys in checkpoint but NOT in model (will be ignored):")
for k in sorted(ck_keys - model_keys):
    print(f"  {k}")

C:\Users\Admin\AppData\Local\Temp\ipykernel_492\3613567431.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ck = torch.load(r"F:\texttt\Output\checkpoints\ep320.pt", map_

Keys in model but NOT in checkpoint (will be random init):

Keys in checkpoint but NOT in model (will be ignored):


## 13. Main Training Loop

In [ ]:
def train(cfg):
    set_seed(cfg.SEED)
    dirs   = setup_folders(cfg.SAVE_DIR)
    device = torch.device(cfg.DEVICE)
    print(f"\nDevice:{device} | T={cfg.T_STEPS} | IMG={cfg.IMG_SIZE}")

    import time

    print("⏳ Building splits...")
    t0 = time.time()
    full_ds, tr_ds, vl_ds, te_ds = build_splits(cfg.DATA_DIR, cfg.IMG_SIZE, seed=cfg.SEED)

    train_indices = list(tr_ds.indices)
    val_indices = list(vl_ds.indices)
    test_indices = list(te_ds.indices)
    train_set = set(train_indices)
    val_set = set(val_indices)
    test_set = set(test_indices)

    assert train_set.isdisjoint(val_set), \
        "Data leakage: train and validation indices overlap!"

    assert train_set.isdisjoint(test_set), \
        "Data leakage: train and test indices overlap!"

    assert val_set.isdisjoint(test_set), \
        "Data leakage: validation and test indices overlap!"

    print(f"  Splits done in {time.time()-t0:.1f}s  |  train={len(tr_ds)}  val={len(vl_ds)}")

    t0 = time.time()
    _img, _lbl = tr_ds[0]
    tr_loader = DataLoader(
        tr_ds, cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
        drop_last=True,
        persistent_workers=(cfg.NUM_WORKERS > 0),
        prefetch_factor=(2 if cfg.NUM_WORKERS > 0 else None),
    )

    ref_gen = torch.Generator()
    ref_gen.manual_seed(cfg.SEED)
    ref_loader = DataLoader(
        vl_ds, cfg.BATCH_SIZE,
        shuffle=True,
        generator=ref_gen,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
        drop_last=False,
        persistent_workers=(cfg.NUM_WORKERS > 0),
        prefetch_factor=(2 if cfg.NUM_WORKERS > 0 else None),
    )
    print(f"Loaders ready  |  tr_batches={len(tr_loader)}  ref_batches={len(ref_loader)}")

    #  Diffusion schedule & perceptual loss
    print("Building diffusion schedule...")
    sched    = build_ddgan_schedule(cfg.T_STEPS, cfg.BETA_MIN, cfg.BETA_MAX, device)
    perc_net = PerceptualLoss().to(device).eval()
    print("Schedule + PerceptualLoss ready")

    # Models
    print("Building models...")
    G     = DenseNet121Generator(cfg.IN_CH, cfg.IN_CH, t_dim=512, z_dim=cfg.Z_DIM).to(device)
    D     = StrongerDiscriminator(cfg.IN_CH, t_dim=512).to(device)
    ema_G = EMA(G, decay=cfg.EMA_DECAY)
    gp    = sum(p.numel() for p in G.parameters() if p.requires_grad)
    dp    = sum(p.numel() for p in D.parameters() if p.requires_grad)
    print(f" G:{gp:,}  D:{dp:,}  ratio:{gp/dp:.1f}:1")


    opt_g = torch.optim.Adam(G.parameters(), lr=cfg.LR_G, betas=(0.5, 0.9))
    opt_d = torch.optim.Adam(D.parameters(), lr=cfg.LR_D, betas=(0.5, 0.9))

    qs       = QualityState(cfg)
    start_ep = 1
    best_fid = float("inf")
    best_ep  = 0
    resumed  = False

    _sched_g_tmp = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt_g, T_max=cfg.EPOCHS, eta_min=cfg.LR_G * 0.05)
    _sched_d_tmp = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt_d, T_max=cfg.EPOCHS, eta_min=cfg.LR_D * 0.05)

    if cfg.RESUME and Path(cfg.RESUME_PATH).exists():
        start_ep, best_fid, best_ep = load_ckpt(
            cfg.RESUME_PATH, G, D, opt_g, opt_d,
            _sched_g_tmp, _sched_d_tmp, qs, device,
            ema=ema_G,
            fallback_lr_g=cfg.LR_G,
            fallback_lr_d=cfg.LR_D,
        )
        resumed = True
        print(f"  Resumed from ep{start_ep-1} (optimizer + scheduler state restored)")
    else:
        print("  Fresh start")

    def apply_freeze(ep):
        for name, param in G.named_parameters():
            if any(k in name for k in ["enc_stem", "enc_block1", "enc_trans1"]):
                param.requires_grad = (ep >= cfg.PHASE2_UNFREEZE_EP)
        return (
            sum(p.numel() for p in G.parameters() if not p.requires_grad),
            sum(p.numel() for p in G.parameters() if p.requires_grad),
        )

    if start_ep < cfg.PHASE2_UNFREEZE_EP:
        fr, tr = apply_freeze(start_ep)
        print(f"  Frozen:{fr:,}  Trainable:{tr:,}")
    else:
        for p in G.parameters(): p.requires_grad = True
        print("  All G parameters trainable")


    warmup_remaining = max(0, cfg.D_WARMUP_EPOCHS - (start_ep - 1))
    if cfg.D_WARMUP_EPOCHS > 0 and warmup_remaining > 0:
        print(f"\n  D warm-up: {warmup_remaining} epoch(s) remaining...")
        bce_wu = nn.BCEWithLogitsLoss()
        sg, pg = qs.get_gap()
        for wu_ep in range(warmup_remaining):
            dl, dc = 0.0, 0
            for imgs, _ in tr_loader:
                imgs = imgs.to(device, non_blocking=True)
                b    = imgs.size(0)
                t    = torch.randint(1, cfg.T_STEPS + 1, (b,), device=device)
                xr, xt, _, _ = q_sample_pair(imgs, t, sched)
                rl = torch.full((b, 1), cfg.REAL_LABEL, device=device)
                fl = torch.full((b, 1), cfg.FAKE_LABEL, device=device)
                with torch.no_grad():
                    x0d = G(xt, torch.randn(b, cfg.Z_DIM, device=device), t,
                            ssim_gap=sg, psnr_gap=pg)
                    xfd = posterior_sample(xt, x0d, t, sched, add_noise=True)
                opt_d.zero_grad(set_to_none=True)
                ld = 0.5 * (bce_wu(D(xr.detach(), xt, t), rl)
                            + bce_wu(D(xfd.detach(), xt, t), fl))
                ld.backward()
                nn.utils.clip_grad_norm_(D.parameters(), 2.0)
                opt_d.step()
                dl += ld.item(); dc += 1
            print(f"    D warm-up {wu_ep+1}/{warmup_remaining}: D_loss={dl/dc:.4f}")
        print("  D warm-up done\n")


    for pg in opt_g.param_groups:
        pg.setdefault("initial_lr", pg["lr"])
    for pg in opt_d.param_groups:
        pg.setdefault("initial_lr", pg["lr"])

    sched_g = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt_g, T_max=cfg.EPOCHS, eta_min=cfg.LR_G * 0.05,
        last_epoch=start_ep - 2)
    sched_d = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt_d, T_max=cfg.EPOCHS, eta_min=cfg.LR_D * 0.05,
        last_epoch=start_ep - 2)
    print(f"  LR schedulers ready — G lr={opt_g.param_groups[0]['lr']:.2e}  "
          f"D lr={opt_d.param_groups[0]['lr']:.2e}")

    print("Loading reference batches (this may take a moment on slow drives)...")
    full_ds.set_mode("eval")

    t0 = time.time()
    prog_src, _  = next(iter(ref_loader))
    print(f"prog_src loaded in {time.time()-t0:.1f}s")
    prog_src     = prog_src[:cfg.PROGRESS_SAMPLES]

    t0 = time.time()
    eval_batch, _ = next(iter(ref_loader))
    print(f"eval_batch loaded in {time.time()-t0:.1f}s")
    eval_batch   = eval_batch[:cfg.ADAPTIVE_EVAL_N]


    full_ds.set_mode("train")

    print("⏳ Shape check...")
    sgi, pgi = qs.get_gap()
    with torch.no_grad():
        dx = torch.randn(2, cfg.IN_CH, cfg.IMG_SIZE, cfg.IMG_SIZE, device=device)
        dz = torch.randn(2, cfg.Z_DIM, device=device)
        dt = torch.randint(1, cfg.T_STEPS + 1, (2,), device=device)
        go = G(dx, dz, dt, ssim_gap=sgi, psnr_gap=pgi)
        assert go.shape == dx.shape, f"G shape mismatch: {go.shape}"
        do = D(dx, dx, dt)
        assert do.shape == (2, 1), f"D shape mismatch: {do.shape}"
    print("  Shape check passed")

    fid_eval = FIDEvaluator(device)

    if resumed and start_ep >= cfg.ADAPTIVE_WARMUP_EP:
        print("⏳ Re-calibrating quality state after resume...")
        full_ds.set_mode("eval")
        sb = eval_batch.to(device, non_blocking=True)
        rsg, rpg = qs.get_gap()
        g01 = ddgan_img2img(ema_G.shadow, sched, cfg, sb, device,
                            ssim_gap=rsg, psnr_gap=rpg)
        mss, mps = compute_ssim_psnr(g01, tensor_to_01(sb))
        
        qs.ssim = mss
        qs.psnr = mps
        
        nsg, npg = qs.get_gap()
        full_ds.set_mode("train")
        print(f"  Re-calibrated: SSIM={mss:.3f}  PSNR={mps:.1f}  "
            f"gap: ssim={nsg:.3f}  psnr={npg:.1f}")

    # ── CSV logs ───
    bce       = nn.BCEWithLogitsLoss()
    train_csv = dirs["logs"] / "train.csv"
    qual_csv  = dirs["logs"] / "quality_feedback.csv"
    fid_csv   = dirs["logs"] / "fid.csv" 
    init_csv(train_csv,
             ["ep", "G", "rec", "perc", "js", "adv", "D", "Gg", "noise_std", "lr_g", "lr_d"])
    init_csv(qual_csv,
             ["ep", "measured_ssim", "measured_psnr", "running_ssim",
              "running_psnr", "ssim_gap", "psnr_gap"])
    init_csv(fid_csv, ["ep", "fid", "is_best"]) 

    print(f"\nTraining {start_ep} → {cfg.EPOCHS}")
    print("=" * 70)
    print(f"Fixed losses: adv={cfg.LAMBDA_ADV}  rec={cfg.LAMBDA_REC}  "
          f"perc={cfg.LAMBDA_PERC}  js={cfg.LAMBDA_JS}")
    print(f"Adaptive: quality gap → G conditioning (every epoch)\n")

    for ep in range(start_ep, cfg.EPOCHS + 1):

        if ep == cfg.PHASE2_UNFREEZE_EP:
            for p in G.parameters(): p.requires_grad = True
            aio = {id(p) for g in opt_g.param_groups for p in g["params"]}
            fp  = [p for p in G.parameters() if id(p) not in aio]
            opt_g.add_param_group({"params": fp, "lr": cfg.LR_G * 0.3})
            print(f"  ep{ep}: ALL G parameters unfrozen")

        sg, pg = qs.get_gap()
        ms, mp = qs.get_measured()
        full_ds.set_mode("train")
        G.train(); D.train()
        noise_std = cfg.D_NOISE_INIT * max(0.0, 1.0 - ep / 450)

        G_sum = rec_sum = perc_sum = adv_sum = js_sum = 0
        D_sum = Gg_sum = nsum = 0
        d_uc  = lazy_r1 = 0
        loss_d = torch.tensor(0.0)


        pbar = tqdm(tr_loader,
                    desc=f"Ep {ep:03d}/{cfg.EPOCHS}  gap_s={sg:.3f} gap_p={pg:.1f}")

        for bidx, (imgs, _) in enumerate(pbar):
            imgs = imgs.to(device, non_blocking=True)
            b    = imgs.size(0)
            t    = torch.randint(1, cfg.T_STEPS + 1, (b,), device=device)
            xr, xt, _, _ = q_sample_pair(imgs, t, sched)
            rl = torch.full((b, 1), cfg.REAL_LABEL, device=device)
            fl = torch.full((b, 1), cfg.FAKE_LABEL, device=device)

            # ── Discriminator step ───────────────────────────────────────
            D.requires_grad_(True)
            opt_d.zero_grad(set_to_none=True)
            with torch.no_grad():
                x0d = G(xt, torch.randn(b, cfg.Z_DIM, device=device), t,
                        ssim_gap=sg, psnr_gap=pg)
                xfd = posterior_sample(xt, x0d, t, sched, add_noise=True)
            if noise_std > 0.0:
                xri = (xr + noise_std * torch.randn_like(xr)).requires_grad_(True)
                xfi = xfd + noise_std * torch.randn_like(xfd)
            else:
                xri = xr.detach().requires_grad_(True)
                xfi = xfd.detach()

            rp   = D(xri, xt, t)
            fp_  = D(xfi, xt, t)
            loss_d = 0.5 * (bce(rp, rl) + bce(fp_, fl))
            lazy_r1 += 1
            if lazy_r1 % cfg.LAZY_REG == 0:
                r1g    = torch.autograd.grad(outputs=rp.sum(), inputs=xri,
                                             create_graph=True)[0]
                loss_d = (loss_d
                          + (cfg.R1_GAMMA / 2) * r1g.pow(2).view(b, -1).sum(1).mean()
                          * cfg.LAZY_REG)
            loss_d.backward()
            nn.utils.clip_grad_norm_(D.parameters(), 2.0)
            opt_d.step()
            D_sum += loss_d.item() * b
            d_uc  += 1

            D.requires_grad_(False)
            opt_g.zero_grad(set_to_none=True)
            z      = torch.randn(b, cfg.Z_DIM, device=device)
            x0_hat = G(xt, z, t, ssim_gap=sg, psnr_gap=pg)
            xf     = posterior_sample(xt, x0_hat, t, sched, add_noise=True)
            la  = bce(D(xf, xt, t), rl)
            lr_ = reconstruction_loss(x0_hat, imgs)
            lp  = perc_net(x0_hat, imgs)
            lj = (
                js_divergence(x0_hat, imgs)
                if cfg.LAMBDA_JS > 0
                else torch.tensor(0.0, device=device)
            )
            loss_g = (cfg.LAMBDA_ADV  * la
                      + cfg.LAMBDA_REC  * lr_
                      + cfg.LAMBDA_PERC * lp
                      + cfg.LAMBDA_JS   * lj)
            loss_g.backward()
            Gg = sum(p.grad.norm().item() ** 2
                     for p in G.parameters() if p.grad is not None) ** 0.5
            nn.utils.clip_grad_norm_(G.parameters(), 10.0)

            Gg_sum += Gg
            opt_g.step()
            ema_G.update(G)

            G_sum    += loss_g.item() * b
            rec_sum  += lr_.item()   * b
            perc_sum += lp.item()    * b
            adv_sum  += la.item()    * b
            js_sum   += lj.item()    * b
            nsum     += b
            pbar.set_postfix(D=f"{loss_d.item():.3f}", adv=f"{la.item():.3f}",
                             rec=f"{lr_.item():.4f}", Gg=f"{Gg:.2f}")

        sched_g.step(); sched_d.step()
        d_loss_ep = D_sum / max(d_uc * cfg.BATCH_SIZE, 1)
        n_batches = len(tr_loader)
        append_csv(train_csv, [
            ep, G_sum/nsum, rec_sum/nsum, perc_sum/nsum, js_sum/nsum,
            adv_sum/nsum, d_loss_ep, Gg_sum/n_batches, noise_std,
            opt_g.param_groups[0]["lr"], opt_d.param_groups[0]["lr"],
        ])

        if ep % cfg.ADAPTIVE_EVAL_EVERY == 0 and ep >= cfg.ADAPTIVE_WARMUP_EP:
            full_ds.set_mode("eval")
            sb  = eval_batch.to(device, non_blocking=True)
            g01 = ddgan_img2img(ema_G.shadow, sched, cfg, sb, device,
                                ssim_gap=sg, psnr_gap=pg)
            mss, mps = compute_ssim_psnr(g01, tensor_to_01(sb))
            qs.update(mss, mps)
            nsg, npg = qs.get_gap()
            nms, nmp = qs.get_measured()
            append_csv(qual_csv, [ep, mss, mps, nms, nmp, nsg, npg])
            print(f"  [Adaptive ep{ep}] SSIM={mss:.3f}  PSNR={mps:.1f}  "
                  f"gap: ssim={nsg:.3f}  psnr={npg:.1f}")
            full_ds.set_mode("train")

        if ep % cfg.PROGRESS_EVERY == 0:
            full_ds.set_mode("eval")
            save_progress(ema_G.shadow, sched, cfg, prog_src, ep,
                          dirs["progress"], device, sg=sg, pg=pg)
            full_ds.set_mode("train")
        if ep % cfg.VIZ_EVERY == 0:
            full_ds.set_mode("eval")
            vb, _ = next(iter(ref_loader))
            save_viz(ema_G.shadow, sched, cfg, vb, ep, dirs["viz"], device,
                     sg=sg, pg=pg)
            full_ds.set_mode("train")

        save_ckpt(dirs["checkpoints"] / "last.pt",
                  ep, G, D, opt_g, opt_d, sched_g, sched_d, qs, best_fid, best_ep, ema=ema_G)
        if ep % cfg.SAVE_EVERY == 0:
            save_ckpt(dirs["checkpoints"] / f"ep{ep:03d}.pt",
                      ep, G, D, opt_g, opt_d, sched_g, sched_d, qs, best_fid, best_ep, ema=ema_G)
            u = disk_usage_gb(cfg.SAVE_DIR)
            print(f"  Disk: {u:.1f} GB")

        if ep % 50 == 0 and ep >= 50:
            full_ds.set_mode("eval")
            fid_eval.reset()
            c, ri = 0, iter(ref_loader)
            while c < 256:
                try: rb, _ = next(ri)
                except StopIteration: ri = iter(ref_loader); rb, _ = next(ri)
                fid_eval.add_real(tensor_to_01(rb)); c += rb.size(0)
            
            
            full_ds.set_mode("train")  
            c, gi = 0, iter(tr_loader)

            while c < 256:
                try: gb, _ = next(gi)
                except StopIteration: gi = iter(tr_loader); gb, _ = next(gi)

                fid_eval.add_fake(
                    ddgan_img2img(
                        ema_G.shadow,
                        sched,
                        cfg,
                        gb.to(device, non_blocking=True),
                        device,
                        ssim_gap=sg,
                        psnr_gap=pg
                    )
                )

                c += gb.size(0)

            full_ds.set_mode("eval")
            fid = fid_eval.compute()
            print(f"  [FID ep{ep}] = {fid:.2f}")
            append_csv(fid_csv, [ep, fid, fid < best_fid]) 

            
            if fid < best_fid:
                best_fid, best_ep = fid, ep
                save_ckpt(dirs["checkpoints"] / "best.pt",
                          ep, G, D, opt_g, opt_d, sched_g, sched_d, qs, best_fid, best_ep, ema=ema_G)
                print(f" New best FID={fid:.2f}")
            full_ds.set_mode("train")

        print(
            f"Ep {ep:03d} | rec={rec_sum/nsum:.4f}  perc={perc_sum/nsum:.4f}  "
            f"js={js_sum/nsum:.5f}  adv={adv_sum/nsum:.4f}  D={d_loss_ep:.4f}  "
            f"Gg={Gg_sum/n_batches:.3f} | gap_s={sg:.3f}  gap_p={pg:.1f}  "
            f"lr={opt_g.param_groups[0]['lr']:.2e}"
        )

    fsg, fpg = qs.get_gap()
    out_dir  = dirs["generated"]
    full_ds.set_mode("eval")
    generate_equal_per_class(ema_G.shadow, sched, cfg, full_ds, out_dir,
                             device, sg=fsg, pg=fpg)
    full_ds.set_mode("train")

    G_eval = ema_G.shadow; G_eval.eval()
    dummy  = torch.randn(1, cfg.IN_CH, cfg.IMG_SIZE, cfg.IMG_SIZE, device=device)
    for _ in range(10):
        ddgan_img2img(G_eval, sched, cfg, dummy, device, ssim_gap=fsg, psnr_gap=fpg)
    torch.cuda.synchronize(); start = time.time()
    for _ in range(100):
        ddgan_img2img(G_eval, sched, cfg, dummy, device, ssim_gap=fsg, psnr_gap=fpg)
        torch.cuda.synchronize()
    ms_val = (time.time() - start) / 100 * 1000
    print(f"\nInference: {ms_val:.1f} ms / image")

    print("\nFinal FID computation...")
    fid_eval.reset(); c = 0; ri = iter(ref_loader); full_ds.set_mode("eval")
    while c < cfg.FINAL_GEN_IMAGES:
        try: imgs, _ = next(ri)
        except StopIteration: ri = iter(ref_loader); imgs, _ = next(ri)
        fid_eval.add_real(tensor_to_01(imgs)); c += imgs.size(0)
    for cd in out_dir.iterdir():
        if cd.is_dir():
            for fp in sorted(cd.glob("*.png")):
                fid_eval.add_fake(
                    transforms.ToTensor()(Image.open(fp).convert("RGB")).unsqueeze(0)
                )
    fid = fid_eval.compute()
    print(f"\n{'='*55}")
    print(f"  FINAL FID  = {fid:.2f}")
    print(f"  Best  FID  = {best_fid:.2f}  at ep{best_ep}")
    print(f"  Images     : {out_dir}")
    print(f"{'='*55}")


print("  train() defined")

if __name__ == "__main__":
    train(cfg)